In [ ]:
"""
scDoRI downstream analysis: patient P1 (PDAC, 11 samples across primary
and metastatic sites).

This script assumes the scDoRI model has already been trained (see
training.py) and picks up from the best checkpoint. It reproduces the
analysis behind every figure actually used in the thesis Results section
(3.1-3.6): Leiden clustering, the topic-Leiden specificity heatmap, the
Topic_7 contamination diagnostics, the classical/basal signature analysis
(across all epithelial cells and within the tumor cluster only),
transcription factor activator/repressor programs, and Hallmark pathway
enrichment.

Note: rna_metacell.X is already log-normalized, so it is not normalized
again here.

"""

import re

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset

from scdori import (
    trainConfig,
    scDoRI,
    load_best_model,
    compute_neighbors_umap,
    compute_topic_gene_matrix,
    plot_topic_activation_heatmap,
    get_top_activators_per_topic,
    get_top_repressor_per_topic,
    compute_activator_tf_activity_per_cell,
    compute_repressor_tf_activity_per_cell,
    get_latent_topics,
    compute_atac_grn_activator_with_significance,
    compute_atac_grn_repressor_with_significance,
)

import decoupler as dc

In [ ]:
# ============================================================
# 0. SETUP - load trained model, latent topics, and GRNs
# ============================================================
# rna_metacell / atac_metacell are assumed already loaded (see
# preprocessing.py / training.py), with tf_indices, onehot_batch, device,
# and num_cells available from the same setup used at training time.

model = load_best_model(
    model, Path(trainConfig.weights_folder_grn) / "best_scdori_best_eval.pth", device
)

n_cells = rna_metacell.n_obs
indices = np.arange(n_cells)
all_dataset = TensorDataset(torch.from_numpy(indices))
all_dataset_loader = DataLoader(
    all_dataset, batch_size=trainConfig.batch_size_cell_prediction, shuffle=False
)

# scDoRI latent topic embedding (used as X_scdori everywhere below)
scdori_latent = get_latent_topics(
    model, device, all_dataset_loader, rna_metacell, atac_metacell,
    num_cells, tf_indices, onehot_batch,
)
rna_metacell.obsm["X_scdori"] = scdori_latent

# ATAC based GRN for activators
grn_act_atac = compute_atac_grn_activator_with_significance(
    model, device, cutoff_val=0.05, outdir="grn_act_atac", num_permutations=trainConfig.num_permutations
)
# ATAC based GRN for repressors
grn_rep_atac = compute_atac_grn_repressor_with_significance(
  model, device, cutoff_val=0.05, outdir="grn_act_atac", num_permutations=trainConfig.num_permutations
)

# Activator/repressor GRNs, precomputed by training.py
# (num_topics x num_tfs x num_genes)
grn_act = np.load("grn_patient/grn_activator__0.05.npy")
grn_rep = np.load("grn_patient/grn_repressor__0.05.npy")
tf_names = rna_metacell.var[rna_metacell.var.gene_type == "TF"].index.values

In [ ]:
# ============================================================
# 3.1 UMAP structure resolves four major cell populations,
#     independently confirmed by unsupervised clustering
# ============================================================
compute_neighbors_umap(
    rna_metacell,
    rep_key="X_scdori",
    umap_n_neighbors=25,
    umap_min_dist=0.1,
    umap_random_state=trainConfig.umap_random_state,
)

# Figure 3.1a: sample / cell type UMAP
sns.set(font_scale=0.3)
sns.set_style("whitegrid")
with plt.rc_context({"figure.figsize": (3, 3), "figure.dpi": 1000}):
    sc.pl.umap(
        rna_metacell,
        color=["extra_sampleName", "Cell Type"],
        add_outline=True,
        outline_color=("white", "black"),
        size=10,
        save="_celltype_sample_umap.png",
    )

# Unsupervised confirmation: Leiden clustering on the scDoRI topic space
sc.tl.leiden(
    rna_metacell,
    resolution=0.1,
)
sc.pl.umap(rna_metacell, color="leiden")

In [ ]:
# ============================================================
# 3.2 Topic_7 and Topic_17 are Leiden-cluster-specific marker
#     topics for the two epithelial subpopulations
# ============================================================
# Figure 3.2a: topic activation heatmap, aggregated by Leiden cluster
df_topic_celltype = plot_topic_activation_heatmap(
    rna_metacell, groupby_key=["leiden"], aggregation="mean",
)

# Figure 3.2b: Topic_7 / Topic_17 activation overlaid on the UMAP
sc.pl.umap(rna_metacell, color=["Topic_7", "Topic_17"])

In [ ]:
# ============================================================
# 3.3 The Topic_7-marked epithelial subpopulation is predominantly
#     contaminating normal lung epithelium, with a distinct
#     liver-derived branch
# ============================================================
def pool_of(sample_name):
    """'poolA_SAMPLE-ID' -> 'poolA'."""
    m = re.match(r"^(pool\w+?)_", sample_name)
    if not m:
        raise ValueError(f"Could not extract pool from sample name: {sample_name!r}")
    return m.group(1)


POOL_METADATA = {
    "poolA": {"label": "T1", "stage": "Primary",    "site": "Pancreas"},
    "poolB": {"label": "T2", "stage": "Primary",    "site": "Pancreas"},
    "poolC": {"label": "T3", "stage": "Primary",    "site": "Other"},
    "poolD": {"label": "M1", "stage": "Metastasis", "site": "Liver"},
    "poolE": {"label": "M2", "stage": "Metastasis", "site": "Liver"},
    "poolF": {"label": "M3", "stage": "Metastasis", "site": "Liver"},
    "poolG": {"label": "M4", "stage": "Metastasis", "site": "Peritoneum"},
    "poolH": {"label": "M5", "stage": "Metastasis", "site": "Lung"},
    "poolI": {"label": "M6", "stage": "Metastasis", "site": "Lung"},
    "poolJ": {"label": "M7", "stage": "Metastasis", "site": "Other"},
    "poolK": {"label": "M7", "stage": "Metastasis", "site": "Other"},
}

SAMPLE_COL = "extra_sampleName"
pools = rna_metacell.obs[SAMPLE_COL].map(pool_of)
missing = sorted(set(pools) - set(POOL_METADATA))
if missing:
    raise ValueError(f"Pools with no metadata entry: {missing}")

rna_metacell.obs["pool"] = pools
rna_metacell.obs["label"] = pools.map(lambda p: POOL_METADATA[p]["label"]).astype("category")
rna_metacell.obs["stage"] = pools.map(lambda p: POOL_METADATA[p]["stage"]).astype("category")
rna_metacell.obs["site"] = pools.map(lambda p: POOL_METADATA[p]["site"]).astype("category")

# Figure 3.3a: site UMAP
sc.pl.umap(rna_metacell, color="site")

In [ ]:
# Which Leiden cluster is the Topic_7-marked one (adjust to match the actual
# label from Figure 3.2a before running the rest of this block)
TOPIC7_CLUSTER = "3"
mask = rna_metacell.obs["leiden"].astype(str) == TOPIC7_CLUSTER
rna_metacell.obs["leiden_group"] = np.where(
    mask, f"leiden {TOPIC7_CLUSTER} (Topic_7)", "other clusters"
)

# Figure 3.3b: pool composition of the Topic_7 cluster
pool_counts = rna_metacell.obs.loc[mask, "pool"].value_counts()
print(f"Pool composition of leiden cluster {TOPIC7_CLUSTER}:")
print(pool_counts)

fig, ax = plt.subplots(figsize=(4, 3))
pool_counts.plot(kind="bar", ax=ax, color="#4C72B0")
ax.set_ylabel("Number of cells")
ax.set_title(f"Pool composition - leiden {TOPIC7_CLUSTER} (Topic_7)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Figure 3.3c: ductal signature score on the UMAP. "Ductal" is a per-cell
# score column that already comes with the dataset's .obs.
if "Ductal" not in rna_metacell.obs.columns:
    raise KeyError("rna_metacell.obs has no 'Ductal' column; it is expected to come with the dataset.")
sc.pl.umap(rna_metacell, color="Ductal")

In [ ]:
# QC check for Section 3.3 (not shown as a figure in the thesis).
# .X is log-normalized, so summed counts are not UMI counts; the number of
# detected genes per cell is unaffected by normalization and is used instead.
X = rna_metacell.X
rna_metacell.obs["n_genes_detected"] = np.asarray((X > 0).sum(axis=1)).ravel()

print(rna_metacell.obs.groupby("leiden_group")["n_genes_detected"].median())

qc_cols = [c for c in ["doublet_score", "n_genes_detected"] if c in rna_metacell.obs.columns]

fig, axes = plt.subplots(1, len(qc_cols), figsize=(4 * len(qc_cols), 4))
if len(qc_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, qc_cols):
    sc.pl.violin(rna_metacell, col, groupby="leiden_group", ax=ax, show=False, rotation=30)
    ax.set_title(col)
plt.tight_layout()
plt.show()

sc.pl.umap(rna_metacell, color=qc_cols, ncols=len(qc_cols), cmap="viridis")

In [ ]:
# Figures 3.3d/3.3e/3.3f: lung-specific normal-epithelial contamination markers
# (3.3d UMAP, 3.3e dotplot scaled per gene, 3.3f violins)
lung_markers = ["SFTPC", "SFTPB", "NAPSA", "NKX2-1"]
present = [g for g in lung_markers if g in rna_metacell.var_names]
missing_markers = [g for g in lung_markers if g not in rna_metacell.var_names]
if missing_markers:
    print("Not found in var_names (check symbol/case):", missing_markers)

if present:
    sc.pl.umap(rna_metacell, color=present, cmap="Reds", ncols=2)
    sc.pl.violin(rna_metacell, present, groupby="leiden_group", rotation=30)
    sc.pl.dotplot(rna_metacell, present, groupby="leiden", standard_scale="var")

In [ ]:
# ============================================================
# 3.4 Topic_17 marks the tumor epithelial population as a whole
#     rather than a position on the classical/basal axis
# ============================================================
signatures = {
    "Tumor_classical": ["AGR3", "KRT20", "TFF2", "GATA6", "LYZ", "TSPAN8",
                         "CLDN18", "TFF1", "CEACAM6", "REG4", "ANXA10"],
    "Tumor_basal": ["FGFBP1", "KRT7", "LY6D", "SERPINB3", "ANXA8", "KRT17",
                     "DHRS9", "SLC2A1", "S100A2"],
    "Normal": ["AMBP", "FXYD2"],
}

# rna_metacell.X is already log-normalized: no normalize_total/log1p here
# (applying them again would transform the data twice).

epi = rna_metacell[rna_metacell.obs["Cell Type"] == "Epithelial cells"].copy()

for name, genes in signatures.items():
    present_genes = [g for g in genes if g in epi.var_names]
    missing_genes = set(genes) - set(present_genes)
    if missing_genes:
        print(f"[{name}] missing from var_names: {missing_genes}")
    sc.tl.score_genes(epi, gene_list=present_genes, score_name=name)

epi_scdori = pd.DataFrame(
    epi.obsm["X_scdori"], index=epi.obs_names,
    columns=[f"Topic_{i}" for i in range(epi.obsm["X_scdori"].shape[1])],
)

TUMOR_CLUSTER = "0"  # leiden cluster of the Topic_17-marked tumor epithelium
in_tumor = (epi.obs["leiden"].astype(str) == TUMOR_CLUSTER).values


def topic_signature_corr(topics, obs):
    return pd.DataFrame({
        "corr_classical": topics.corrwith(obs["Tumor_classical"]),
        "corr_basal": topics.corrwith(obs["Tumor_basal"]),
    }).sort_values("corr_basal", ascending=False)


# Per-topic correlation with the classical/basal signature scores,
# (a) across all epithelial cells and (b) within the tumor cluster only
topic_corr = topic_signature_corr(epi_scdori, epi.obs)
print("All epithelial cells:")
print(topic_corr.round(3))

topic_corr_tumor = topic_signature_corr(epi_scdori[in_tumor], epi.obs[in_tumor])
print(f"Tumor cluster (leiden {TUMOR_CLUSTER}) only:")
print(topic_corr_tumor.round(3))

In [ ]:
# Stratify epithelial cells into classical-only / basal-only / hybrid groups
# (median split on both scores simultaneously; signature-low cells are not
# shown) and compare Topic_17 activation across groups, again across all
# epithelial cells and within the tumor cluster only
median_classical = epi.obs["Tumor_classical"].median()
median_basal = epi.obs["Tumor_basal"].median()

hybrid = epi.obs[(epi.obs["Tumor_classical"] > median_classical) & (epi.obs["Tumor_basal"] > median_basal)]
pure_classical = epi.obs[(epi.obs["Tumor_classical"] > median_classical) & (epi.obs["Tumor_basal"] <= median_basal)]
pure_basal = epi.obs[(epi.obs["Tumor_classical"] <= median_classical) & (epi.obs["Tumor_basal"] > median_basal)]

print(f"Hybrid (high both): {len(hybrid)}")
print(f"Pure classical: {len(pure_classical)}")
print(f"Pure basal: {len(pure_basal)}")

epi.obs["phenotype_group"] = "Other"
epi.obs.loc[hybrid.index, "phenotype_group"] = "Hybrid"
epi.obs.loc[pure_classical.index, "phenotype_group"] = "Pure_classical"
epi.obs.loc[pure_basal.index, "phenotype_group"] = "Pure_basal"
epi.obs["phenotype_group"] = pd.Categorical(
    epi.obs["phenotype_group"],
    categories=["Pure_classical", "Hybrid", "Pure_basal"],
)

subset = epi[epi.obs["phenotype_group"].isin(["Pure_classical", "Hybrid", "Pure_basal"])]
sc.pl.violin(subset, keys="Topic_17", groupby="phenotype_group", rotation=30)

print("Topic_17 by group, all epithelial cells:")
print(epi.obs.groupby("phenotype_group", observed=True)["Topic_17"].agg(["count", "mean", "median"]))

print(f"Topic_17 by group, tumor cluster (leiden {TUMOR_CLUSTER}) only:")
print(epi.obs.loc[in_tumor].groupby("phenotype_group", observed=True)["Topic_17"].agg(["count", "mean", "median"]))

In [ ]:
# ============================================================
# 3.5 Transcription factor programs underlying Topic_17
# ============================================================
# Per-topic top activator / repressor TFs (z-scored activity), across all
# topics - this is what df_topic_activator / df_topic_repressor feed into
# the Topic_7/Topic_17-specific heatmap below
df_topic_activator, top_regulators = get_top_activators_per_topic(
    grn_act, tf_names, scdori_latent,
    selected_topics=None, top_k=2, clamp_value=1e-8, zscore=True,
    figsize=(25, 10), out_fig="tf_activators_per_topic.png",
)

df_topic_repressor, top_regulators_repressor = get_top_repressor_per_topic(
    grn_rep, tf_names, scdori_latent,
    selected_topics=None, top_k=20, clamp_value=1e-8, zscore=True,
    figsize=(25, 10), out_fig="tf_repressors_per_topic.png",
)

In [ ]:
# Figure 3.5a(i)/(ii): activator / repressor heatmaps restricted to Topic_7
# and Topic_17 (Topic_7 shown alongside for completeness, not interpreted
# in the context of PDAC tumor biology given Section 3.3)
top_k = 10
topics_of_interest = ["Topic_7", "Topic_17"]


def top_tf_heatmap(df_topic, topics, top_k, title, cmap):
    sub = df_topic.loc[topics].T  # TFs x topics
    top_tfs = sorted(set().union(*[set(sub[t].nlargest(top_k).index) for t in topics]))
    sub = sub.loc[top_tfs]

    fig, ax = plt.subplots(figsize=(4, max(4, 0.35 * len(top_tfs))))
    sns.heatmap(sub, cmap=cmap, annot=True, fmt=".2f", ax=ax,
                cbar_kws={"label": title})
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
    return sub


act_sub = top_tf_heatmap(df_topic_activator, topics_of_interest, top_k,
                          "Activator activity (z)", "Reds")
rep_sub = top_tf_heatmap(df_topic_repressor, topics_of_interest, top_k,
                          "Repressor activity (z)", "Blues")

In [ ]:
# Figure 3.5b: per-cell activity for the TFs of interest on the UMAP.
# Per-cell scores combine each cell's Topic_7/Topic_17 activation with the
# topic-level TF scores, so they are derived from the same scores as
# Figure 3.5a rather than being an independent test.
act_activity = compute_activator_tf_activity_per_cell(
    grn_act, tf_names, scdori_latent, selected_topics=[7, 17], zscore=True,
)
rep_activity = compute_repressor_tf_activity_per_cell(
    grn_rep, tf_names, scdori_latent, selected_topics=[7, 17], zscore=True,
)

tf_names_list = list(tf_names)


def add_tf_activity(adata, activity_matrix, tf, colname):
    idx = tf_names_list.index(tf)
    adata.obs[colname] = activity_matrix[:, idx]


add_tf_activity(rna_metacell, act_activity, "KLF5", "KLF5_activator_activity")
add_tf_activity(rna_metacell, act_activity, "SOX4", "SOX4_activator_activity")
add_tf_activity(rna_metacell, rep_activity, "GLIS3", "GLIS3_repressor_activity")
add_tf_activity(rna_metacell, rep_activity, "HMGA1", "HMGA1_repressor_activity")
add_tf_activity(rna_metacell, rep_activity, "MXI1", "MXI1_repressor_activity")
add_tf_activity(rna_metacell, rep_activity, "RBPJ", "RBPJ_repressor_activity")

tf_cols = [
    "KLF5_activator_activity", "SOX4_activator_activity",
    "GLIS3_repressor_activity", "HMGA1_repressor_activity",
    "MXI1_repressor_activity", "RBPJ_repressor_activity",
]

sc.pl.umap(rna_metacell, color=tf_cols, cmap="RdBu_r", ncols=3)

In [ ]:
# ============================================================
# 3.6 Pathway enrichment characterizes the biological program
#     of Topic_17
# ============================================================
topic_gene_embedding = compute_topic_gene_matrix(model, device)
adata_gene = sc.AnnData(topic_gene_embedding)
adata_gene.obs.index = [f"Topic_{i}" for i in range(model.num_topics)]
adata_gene.var.index = [g.upper() for g in rna_metacell.var.index]
adata_gene.raw = adata_gene

msigdb = dc.op.resource("MSigDB")
hallmark = msigdb[msigdb["collection"] == "hallmark"]
genesets = hallmark[~hallmark.duplicated(["geneset", "genesymbol"])]
genesets = genesets.rename(columns={"geneset": "source", "genesymbol": "target"})

# Only the Hallmark collection is used (HPO and GO collections were not
# informative in this context)
dc.mt.ora(adata_gene, genesets, tmin=5, verbose=True)
ora_scores = adata_gene.obsm["score_ora"]  # topics x genesets
ora_padj = adata_gene.obsm["padj_ora"]

for topic in topics_of_interest:
    top = ora_scores.loc[topic].sort_values(ascending=False).head(15)
    sig = ora_padj.loc[topic, top.index]
    print(topic)
    print(pd.DataFrame({"score": top, "padj": sig}))

In [ ]:
# Figure 3.6: top enriched Hallmark gene sets, Topic_7 vs. Topic_17
top_n = 10
top_pathways = sorted(set(
    p for t in topics_of_interest
    for p in ora_scores.loc[t].sort_values(ascending=False).head(top_n).index
))
heat = ora_scores.loc[topics_of_interest, top_pathways]

fig, ax = plt.subplots(figsize=(6, max(4, 0.35 * len(top_pathways))))
im = ax.imshow(heat.T.values, aspect="auto", cmap="RdBu_r")
ax.set_xticks(range(len(topics_of_interest)))
ax.set_xticklabels(topics_of_interest)
ax.set_yticks(range(len(top_pathways)))
ax.set_yticklabels(top_pathways, fontsize=7)
fig.colorbar(im, ax=ax, label="ORA score")
plt.title("Top pathways per topic (Hallmark)")
plt.tight_layout()
plt.savefig("fig_pathway_final.png", dpi=300, bbox_inches="tight")
plt.show()